# 00 — Setup & Data Audit

Verifies the Kaggle Amazon Reviews dataset is present, prints schema/shape/distribution stats, and saves the first descriptive figures. **Run before anything else.**

Expected file: `data/raw/train.csv` from <https://www.kaggle.com/datasets/kritanjalijain/amazon-reviews>. Schema: `polarity` (1=neg, 2=pos), `title`, `text`.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

from utils import RAW_DIR, FIGURES_DIR, find_amazon_csv, set_seed

sns.set_theme(style="whitegrid")
set_seed()

In [ ]:
csv_path = find_amazon_csv()
if csv_path is None:
    raise FileNotFoundError(
        f"Could not find train.csv under {RAW_DIR}. "
        "Download the Kaggle dataset and unzip into data/raw/."
    )
print(f"Loading {csv_path}")

In [ ]:
# Polars handles the file fast and lazily; the Kaggle CSV has no header row.
df = pl.read_csv(csv_path, has_header=False, new_columns=["polarity", "title", "text"])
print("Shape:", df.shape)
print("Schema:", df.schema)
df.head(3)

In [ ]:
print("Polarity counts:")
print(df["polarity"].value_counts().sort("polarity"))
print()
print("Null counts:")
print(df.null_count())

In [ ]:
# Length distributions
df = df.with_columns([
    df["title"].fill_null("").str.len_chars().alias("title_len"),
    df["text"].fill_null("").str.len_chars().alias("text_len"),
    df["text"].fill_null("").str.split(" ").list.len().alias("text_words"),
])
print(df.select(["title_len", "text_len", "text_words"]).describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

pol_counts = df["polarity"].value_counts().sort("polarity").to_pandas()
sns.barplot(data=pol_counts, x="polarity", y="count", ax=axes[0], color="#3b7dd8")
axes[0].set_title("Polarity distribution")
axes[0].set_xlabel("polarity (1 = negative, 2 = positive)")

# Cap at 1500 chars for a readable histogram
text_len_pd = df["text_len"].to_pandas().clip(upper=1500)
sns.histplot(text_len_pd, bins=60, ax=axes[1], color="#d87a3b")
axes[1].set_title("Review length (chars, capped at 1500)")
axes[1].set_xlabel("characters")

plt.tight_layout()
fig_path = FIGURES_DIR / "00_data_audit.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved {fig_path}")

## Findings to record in the LaTeX Dataset section

- Actual row count: _fill from `df.shape` above_
- Polarity balance: _fill from value_counts above_
- Median review length (chars / words): _fill from describe()_
- Reconcile against project brief (which states ~34.7M rows). The canonical kritanjalijain Kaggle entry is the **Zhang et al. (2015) 4M binary polarity subset** — if the count above is ~4M, this is expected; if ~34M, you have a different mirror.
